In [1]:

import os
from pathlib import Path
in_folder_path = Path('/kaggle/input/braintumor-train-pl')
scripts_dir = Path(in_folder_path / 'scripts')
from tqdm import tqdm



In [2]:
from torch.utils.data import DataLoader, Dataset
from torch import nn
from torchvision import transforms
import torch
import pandas as pd

# package_path = "../input/efficientnet-pytorch/EfficientNet-PyTorch/EfficientNet-PyTorch-master/"
import sys 
# sys.path.append(package_path)

# import efficientnet_pytorch
import glob
import numpy as np
import pydicom
import cv2
import pytorch_lightning as pl
from transformers import DeiTFeatureExtractor, DeiTForImageClassification, AutoConfig


2026-04-28 09:06:12.834037: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777367172.847286      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777367172.851355      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 09:06:12.868153: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:

os.chdir(scripts_dir)
exec(Path("dataset.py").read_text())
exec(Path("model.py").read_text())
os.chdir('/kaggle/working')

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/braintumor-train-pl/scripts'

In [4]:
test_df = pd.read_csv("../input/rsna-miccai-brain-tumor-radiogenomic-classification/sample_submission.csv")
models_folder_path = Path(in_folder_path / 'models')
models_preds = []
n_models = 5

backbone = torch.load('../input/braintumor-train-pl/base_model.pt')
base_model = Model(backbone)

feats_extractor = torch.load('../input/braintumor-train-pl/feats_extractor.pt')
models_dir = Path('../input/braintumor-train-pl/models')

models = [
    '../input/braintumor-train-pl/models/model0_epoch=3_roc_auc=0.585.ckpt',
    '../input/braintumor-train-pl/models/model1_epoch=2_roc_auc=0.526.ckpt',
    '../input/braintumor-train-pl/models/model2_epoch=2_roc_auc=0.663.ckpt',
    '../input/braintumor-train-pl/models/model3_epoch=1_roc_auc=0.527.ckpt',
    '../input/braintumor-train-pl/models/model4_epoch=4_roc_auc=0.494.ckpt'
    
]
for i, model_name in enumerate(models):
    print(f'Inference#[{i}/{n_models}] {model_name}')
    
    test_ds = DataRetriever(
        test_df["BraTS21ID"].values,        
        '../input/rsna-miccai-brain-tumor-radiogenomic-classification/test'
   )
    
    test_dl = DataLoader(
        test_ds,
        batch_size=4,
        shuffle=False,
        num_workers=8,
        collate_fn=FeatureExtractorCollate(feats_extractor)
    )
    
    trainer = pl.Trainer(gpus=1, num_sanity_val_steps=0)
    state_dict = torch.load(model_name)['state_dict']
    base_model.load_state_dict(state_dict)

    preds = trainer.predict(base_model, test_dl)
    flat_preds = []
    for p in preds:
        flat_preds += p.flatten().tolist()
    
    models_preds.append(flat_preds)

FileNotFoundError: [Errno 2] No such file or directory: '../input/braintumor-train-pl/base_model.pt'

In [5]:
avg_preds = np.array(models_preds).mean(axis=0)
avg_preds.shape

/tmp/ipykernel_11/3351833891.py:1: RuntimeWarning: Mean of empty slice.
  avg_preds = np.array(models_preds).mean(axis=0)
/usr/local/lib/python3.11/dist-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


()

In [6]:
submission = pd.DataFrame({"BraTS21ID": test_df.BraTS21ID, "MGMT_value": avg_preds})
submission.to_csv("submission.csv", index=False)
submission.head(30)

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,BraTS21ID,MGMT_value
0,356,NaN
1,140,NaN
2,757,NaN
3,275,NaN
4,570,NaN
5,155,NaN
6,740,NaN
7,21,NaN
8,84,NaN
9,729,NaN
